In [1]:
import pandas as pd
import os
import io
import re
from io import BytesIO
# from ftplib import FTP
import paramiko 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# def ConnectFTP(server, username, password):
#     """
#     Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
#     Parametros:
#     server (str): La direccion del servidor FTP.
#     username (str): El nombre de usuario para la conexion FTP.
#     password (str): La contrasena para la conexion FTP.
#     Retorna: Un objeto FTP conectado al servidor.
#     """
#     ftp = FTP(timeout=60)                
#     ftp.connect(server, 21)               
#     ftp.login(user=username, passwd=password)

#     ftp.set_pasv(True)                    
#     ftp.voidcmd("TYPE I")                 

#     print(f"Conectado a {server}")
#     return ftp

# def UploadCsvToFTP(df, path):
#     """
#     Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
#     Parametros:
#     df (DataFrame): El DataFrame que se desea subir.
#     path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
#     Retorna: None
#     """
#     csv_buffer = io.BytesIO()
#     df.to_csv(csv_buffer, index=False, encoding='utf-8')
#     csv_buffer.seek(0)
#     ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
#     # remote_path = f"/pruebas/csv/{path}"
#     ftp.storbinary(f"STOR {path}", csv_buffer)
#     ftp.quit()
#     print("Archivo subido correctamente:", path)

# def ReadCsvFromFTP(remote_file_path):
#     '''
#     Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
#     Parametros:
#     ftp (FTP): Un objeto FTP conectado al servidor.
#     remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
#     Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
#     '''
#     ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
#     with BytesIO() as bio:
#         ftp.retrbinary(f'RETR {remote_file_path}', bio.write)
#         bio.seek(0)
#         df = pd.read_csv(bio)
#     return df

def ConnectSFTP(server, username, password, port=22):
    """
    Establece una conexión SFTP mediante SSH.

    Retorna:
        ssh: cliente SSH necesario para cerrar correctamente la conexión.
        sftp: cliente utilizado para navegar y transferir archivos.
    """
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    ssh.connect(
        hostname=server,
        port=port,
        username=username,
        password=password,
        timeout=60,
    )

    sftp = ssh.open_sftp()

    print(f"Conectado mediante SFTP a {server}:{port}")
    return ssh, sftp

def UploadCsvToSFTP(df, path):
    """
    Convierte un DataFrame a CSV en memoria y lo sube mediante SFTP.
    """
    csv_buffer = io.BytesIO()
    df.to_csv(
        csv_buffer,
        index=False,
        encoding="utf-8"
    )
    csv_buffer.seek(0)
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    # remote_path = f"riecs/pruebas2/csv/{path}"
    try:
        sftp.putfo(csv_buffer, path)
        print("Archivo subido correctamente:", path)
    finally:
        sftp.close()
        ssh.close()

def ReadCsvFromSFTP(remote_file_path):
    """
    Lee un archivo CSV desde un servidor SFTP y lo carga
    en un DataFrame de pandas.

    Parámetros:
        remote_file_path (str):
            Ruta completa del archivo CSV dentro del servidor.

    Retorna:
        pandas.DataFrame:
            DataFrame con el contenido del archivo CSV.
    """
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    try:
        with sftp.open(remote_file_path, mode="rb") as remote_file:
            df = pd.read_csv(remote_file)
        return df
    finally:
        sftp.close()
        ssh.close()

In [3]:
def Group_sust(x):
    '''
    Descripcion: Esta funcion agrupa los identificadores de sustancias en categorías específicas.
    Parametros:
        x (int): Identificador de la sustancia.
    Retorna: Una cadena que representa la categoría de la sustancia.
    '''
    dict_sust = {'Alcohol':(3,4,5,84),'Tabaco':(1,2), 'Marihuana':(6,7,8,9), 'Cocaína':(12,13,14,15,86) , 'Metanfetaminas': (16,17,18,20,48,85) ,'Opioides': (57,58,59,60,61,62,63,64,65,66,67),'Inhalables':(24,25,26,27,28,76), 'Alucinógenos':(29,31,32,33,34,35,36,37,38,41) ,'Medicamentos':(11,21,40,52,53,55,68,69,70,71,72,73,74,75,77,80),'Nuevas Sustancias Psicoactivas':(10,19,22,23,30,39,42,43,44,45,46,47,49,50,51,81),'Otras Sustancias':(54,56,78,79,82), 'ProblemasDeSaludMental': (83, 83000), 'Ninguna': (10000, 10000000)} 
    for key, value in dict_sust.items():
        if x in value:
            return key

def NoGroup_sust(x):
    '''
    Descripcion: Esta funcion devuelve el nombre de la sustancia basado en su identificador.
    Parametros:
        x (int): Identificador de la sustancia.
    Retorna: Una cadena que representa el nombre de la sustancia.
    '''
    dict_sust = {
        'Tabaco': [1],
        'Nicotina en dispositivo electrónico': [2],
        'Bebidas Fermentadas': [3, 84],
        'Bebidas Destiladas': [4],
        'Alcohol puro (alcohol del 96)': [5],
        'Mariguana estándar': [6],
        'Mariguana modificada, con alto nivel de THC': [7],
        'Hachís - Resina': [8],
        'Hachís - Aceite o miel': [9],
        'Mariguana sintética': [10],
        'Medicamentos con compuestos cannábicos': [11],
        'Cocaína en polvo blanco': [12, 86],
        'Crack': [13],
        'Basuco (pasta base de cocaína)': [14],
        'Otras presentaciones de cocaína': [15],
        'Metanfetaminas(cristal)': [16, 85],
        'Metanfetaminas(hielo)': [17],
        'MetanfetaminasMetas': [18],
        'Sales de baño': [19],
        'Anfetaminas y derivados': [20],
        'Anorexígenos': [21],
        'Drogas de diseño con efectos estimulantes': [22],
        'Otros estimulantes': [23],
        'Solventes o removedores': [24],
        'Pegamentos': [25],
        'Esmaltes y pinturas': [26],
        'Gasolinas y combustibles': [27],
        'Otros inhalantes': [28],
        'LSD': [29],
        'LSA': [30],
        'Hongos alucinógenos': [31],
        'Peyote (mescalina)': [32],
        'Salvia divinorum u otras variedades de salvia': [33],
        'Ololiuhqui': [34],
        'Floripondio': [35],
        'Ayahuasca': [36],
        'Otras plantas con efectos alucinógenos o disociativos': [37],
        'Fenciclidina (PCP, polvo de ángel)': [38],
        'Análogos del PCP': [39],
        'Ketamina': [40],
        'DMT': [41],
        'Triptaminas psicodélicas derivadas y análogas del DMT': [42],
        'Grupo 2Cx': [43],
        'Grupo DOx': [44],
        'Grupo NBOMe': [45],
        'Otras drogas de diseño con efectos psicodélicos': [46],
        'Otras sustancias con efectos psicodélicos': [47],
        'Éxtasis (MDMA)': [48],
        'Drogas de diseño con efectos similares al éxtasis': [49],
        'Piperazinas con efectos similares al éxtasis': [50],
        'Aminoindanos': [51],
        'Benzodiacepinas y sustancias relacionadas': [52],
        'Rohypnol': [53],
        'GHB o similares': [54],
        'Sedantes hipnóticos': [55],
        'Otros depresores': [56],
        'Opio': [57],
        'Morfina(uso fuera de prescripción)': [58],
        'Codeína(uso fuera de prescripción)': [59],
        'Heroína blanca': [60],
        'Heroína negra': [61],
        'Oxicodona': [62],
        'Opiodes sintéticos (uso fuera de prescripción)': [63],
        'Fentanilo y análogos (uso fuera de prescripción)': [64],
        'Dextrometorfano (uso fuera de prescripción)': [65],
        'Derivados opiodes combinados con analgésicos': [66],
        'Desomorfina': [67],
        'Anticolinérgicos': [68],
        'Antiespasmódicos': [69],
        'Antiparkinsonianos': [70],
        'Anticonvulsivos': [71],
        'Antidepresivos': [72],
        'Antipsicóticos': [73],
        'Antihistamínicos': [74],
        'Otras sustancias de abuso con utilidad médica': [75],
        'Nitritos': [76],
        'Anestésicos': [77],
        'Bebidas energizantes': [78],
        'Bebidas inteligentes': [79],
        'Esteroides anabólicos': [80],
        'Sustancias no clasificadas': [81],
        'Sustancias sin especificar': [82],
        'Problemas de salud mental': [83],
        'Ninguna': [10000]
    }
    for sust, codigos in dict_sust.items():
        if x in codigos:
            return sust
        
def ModSustancias(df):
    '''
    Descripcion: Esta funcion agrupa las sustancias en categorías específicas y crea dos DataFrames:
    uno con las sustancias agrupadas y otro con las sustancias sin agrupar.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene los datos de las sustancias.
    Retorna: Dos DataFrames, uno con las sustancias agrupadas y otro con las sustancias sin agrupar.
    '''
    dfsust = df.copy()
    dfgroup = df.copy()
    sustancia_columns = [col for col in df.columns if col.startswith('SustanciaId')]
    for col in sustancia_columns:
        dfgroup[col] = dfgroup[col].apply(Group_sust)
        dfsust[col] = dfsust[col].apply(NoGroup_sust)
    dfgroup['MayorImpactoSustanciaId'] = dfgroup['MayorImpactoSustanciaId'].apply(Group_sust)
    dfsust['MayorImpactoSustanciaId'] = dfsust['MayorImpactoSustanciaId'].apply(NoGroup_sust)
    return dfgroup, dfsust

def Fecha(df):
    '''
    Descripcion: Esta funcion convierte la columna 'FechaRegistro' del DataFrame a un formato de fecha y extrae el año, mes y semestre.
    Parametros:
    df (DataFrame): DataFrame de pandas que contiene la columna 'FechaRegistro'.
    Retorna: Un DataFrame con las columnas 'Año', 'Mes' y 'Semestre' añadidas.
    '''
    df['FechaRegistro'] = pd.to_datetime(df['FechaRegistro'])
    df['Año'] = df['FechaRegistro'].dt.year
    df['Mes'] = df['FechaRegistro'].dt.month
    df['Semestre'] = df['Mes'].apply(lambda x: 1 if x in range(1, 7) else 2)
    df['Semestre'] = df['Año'].astype(str) + '-' + df['Semestre'].astype(str).str.zfill(2)
    return df

def Escolar(x):
    '''
    Descripcion: Esta funcion categoriza el nivel educativo basado en un código numérico.
    Parametros:
    x (int): Código del nivel educativo.
    Retorna: Una cadena que representa el nivel educativo.
    '''
    if x == 1:
        return 'Sin Estudios'
    elif x == 2 or x == 3:
        return 'Primaria'
    elif x == 4 or x == 5:
        return 'Secundaria'
    elif x in range(6, 10):
        return 'Preparatoria o Carrera Técnica'
    elif x == 10 or x == 11:
        return 'Estudios Superiores'
    elif x == 12 or x == 13:
        return 'Estudios de Posgrado'

def EstCiv(x):
    '''
    Descripcion: Esta funcion categoriza el estado civil basado en un código numérico.
    Parametros:
    x (int): Código del estado civil.
    Retorna: Una cadena que representa el estado civil.
    '''
    if x == 1:
        return 'Soltero(a)'
    elif x == 2:
        return 'Casado(a)'
    elif x == 3:
        return 'Unión Libre'
    elif x == 4:
        return 'Separado(a)'
    elif x == 5:
        return 'Divorciado(a)'
    elif x == 6:
        return 'Viudo(a)'

def Ocupacion(x):
    '''
    Descripcion: Esta funcion categoriza la ocupación laboral basada en un código numérico.
    Parametros:
    x (int): Código de la ocupación laboral.
    Retorna: Una cadena que representa la ocupación laboral.
    '''
    if x in range(1, 4):
        return 'Estudiante'
    elif x == 4:
        return 'Con actividad laboral'
    elif x == 5:
        return 'Hogar'
    elif x == 6:
        return 'Pensionado o jubilado'
    elif x == 7:
        return 'Sin ocupacion'
    elif x == 8:
        return 'Sin Dato'

def EstratoSocial(x):
    '''
    Descripcion: Esta funcion categoriza el estrato social basado en un código numérico.
    Parametros:
    x (int): Código del estrato social.
    Retorna: Una cadena que representa el estrato social.
    '''
    if x == 1:
        return 'Alto'
    elif x == 2 :
        return 'Medio Alto'
    elif x == 3:
        return 'Medio Bajo'
    elif x == 4:
        return 'Bajo'
    elif x == 5:
        return 'Muy Bajo'

def LGBTI(x):
    '''
    Descripcion: Esta funcion categoriza la identidad de género o la orientación sexual basada en un código numérico.
    Parametros:
    x (int): Código de la identidad de género o la orientación sexual.
    Retorna: Una cadena que representa la identidad de género o la orientación sexual.
    '''
    if x == 1:
        return 'Lesbica'
    elif x == 2:
        return 'Gay'
    elif x == 3:
        return 'Bisexual'
    elif x == 4:
        return 'Transgenero hombre'
    elif x == 5:
        return 'Transgenero mujer'
    elif x == 6:
        return 'Transexual hombre'
    elif x == 7:
        return 'Transexual mujer'
    elif x == 8:
        return 'Travesti hombre'
    elif x == 9:
        return 'Travesti mujer'
    elif x == 10:
        return 'Intersexual'
    elif x == 11:
        return 'Otro'

def Sexo(x):
    '''
    Descripcion: Esta funcion categoriza el sexo basado en un código numérico.
    Parametros:
    x (int): Código del sexo.
    Retorna: Una cadena que representa el sexo.
    '''
    if x == 1:
        return 'Hombre'
    elif x == 2:
        return 'Mujer'

def Edad_Quincenal(x):
    '''
    Descripcion: Esta funcion categoriza la edad en rangos quincenales.
    Parametros:
    x (int): Edad en años.
    Retorna: Una cadena que representa el rango de edad quincenal.
    '''
    if x < 10:
        return '0-9'
    elif x < 15:
        return '10-14'
    elif x < 20:
        return '15-19'
    elif x < 25:
        return '20-24'
    elif x < 30:
        return '25-29'
    elif x < 35:
        return '30-34'
    elif x < 40:
        return '35-39'
    elif x < 45:
        return '40-44'
    elif x < 50:
        return '45+'

def Recod_mot (x):
    '''
    Descripcion: Esta funcion recodifica los motivos de consulta basados en un código numérico.
    Parametros:
        x (int): Código del motivo de consulta.
    Retorna: Una cadena que representa el motivo de consulta.
    '''
    dictmotv = {'ConsumoDeDrogas': 1, 'ConsumoDeBebidasAlcoholicas': 2, 'ConsumoDeTabaco': 3, 'Ludopatia': 4, 'Otro': 5, 'Depresion': 6, 'Psicosis': 7, 'Epilepsia': 8, 'TrastornosMentales': 9, 'Demencia': 10, 'Autolesion': 11, 'Suicidio': 12, 'Ansiedad': 13}
    for key, value in dictmotv.items():
        if x == value:
            return key
        
def MotivoConsultas(df):
    '''
    Descripcion: Esta funcion recodifica los motivos de consulta en un DataFrame y aplica una función de recodificación a cada columna de motivo.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene las columnas de motivos de consulta.
    Retorna: Un DataFrame con las columnas de motivos de consulta recodificadas.
    '''
    lista_motivos = [col for col in df.columns if re.search(r'Motivo_\d+', col)]
    for col in lista_motivos:
        df[col] =df[col].apply(Recod_mot)
    return df

def MotivoConsulta2(df):
    '''
    Descripcion: Esta funcion crea columnas binarias para cada motivo de consulta en un DataFrame.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene las columnas de motivos de consulta.
    Retorna: Un DataFrame con columnas binarias para cada motivo de consulta.
    '''
    lista_motivos = [col for col in df.columns if re.search(r'Motivo_\d+', col)]
    valores_a_buscar = ['ConsumoDeDrogas', 'ConsumoDeBebidasAlcoholicas', 'ConsumoDeTabaco','Ludopatia', 'Otro', 'Depresion', 'Psicosis', 'Epilepsia','TrastornosMentales', 'Demencia', 'Autolesion', 'Suicidio', 'Ansiedad']

    for valor in valores_a_buscar:
        df[valor] = 0
        
    for col in lista_motivos:
        for valor in valores_a_buscar:
            df[valor] = df[valor] | df[col].apply(lambda x: 1 if pd.notna(x) and valor in str(x) else 0)
    
    return df

def CentroCostoDet ():
    df1 = ReadCsvFromSFTP (r'riecs/pruebas2/csv/Moddata/CentrosDeCosto.csv')
    df2 = ReadCsvFromSFTP (r'riecs/pruebas2/csv/Moddata/CentrosDeCostoEstado.csv')
    dict1 = dict(zip(df1['CentroCostoId'], df1['CentroCostoClave']))
    dict2 = dict(zip(df2['CENTRO'], df2['ESTADO']))
    dict3 = {'CIUDAD DE MÉXICO':"CDMX", 'ESTADO DE MÉXICO': "EMEX", 'JALISCO':"JAL", 'SINALOA': 'SIN', 'BAJA CALIFORNIA':"BC", 'CHIHUAHUA': "CHH", 'GUANAJUATO': 'GTO', 'QUINTANA ROO':"QNTROO", 'COAHUILA':  "COAH", 'NUEVO LEÓN':  "NVL", 'MICHOACÁN': "MICH", 'GUERRERO': "GRO", 'COLIMA': "COL", 'BAJA CALIFORNIA SUR':"BCS", 'TAMAULIPAS': "TAM",
        'VERACRUZ': "VRC", 'SONORA': "SON", 'PUEBLA': "PBL", 'DURANGO': "DUR", 'AGUASCALIENTES':  "AGS", 'YUCATÁN':"YCT", 'HIDALGO':"HDO",'ZACATECAS':"ZAC", 'QUERÉTARO': "QRO", 'SAN LUIS POTOSÍ': "SLP" , 'OAXACA': "OAX", 'TABASCO': "TBS", 'CHIAPAS': "CHPS", 'MORELOS':"MOR", 'TLAXCALA': "TLX", 'CAMPECHE':"CAM", 'NAYARIT':"NAY"}
    return dict1, dict2, dict3

def Mod_SocioDem (df):
    df = Fecha(df)
    df['ComunEscolaridadId'] = df['ComunEscolaridadId'].apply(Escolar)
    df = MotivoConsultas(df)
    df = MotivoConsulta2(df)
    df['ComunEstadoCivilId'] = df['ComunEstadoCivilId'].apply(EstCiv)
    df['ComunOcupacionId'] = df['ComunOcupacionId'].apply(Ocupacion)
    df['ComunEstratoSocialId'] = df['ComunEstratoSocialId'].apply(EstratoSocial)
    df['SexoId'] = df['SexoId'].apply(Sexo)
    df['ComunComunidadLGBTTTIId'] = df['ComunComunidadLGBTTTIId'].apply(LGBTI)
    df['Edad_Años'] = df['Edad']
    df['Edad'] = df['Edad'].apply(Edad_Quincenal)
    dict1, dict2, dict3 = CentroCostoDet()
    df['Estado'] = df['CentroCostoId'].map(dict1).map(dict2).map(dict3)
    return df

def RecorridoSustancia(df, sustancias, excluir_mayor_impacto=None):
    '''
    Descripcion: Esta funcion filtra un DataFrame para incluir solo las filas donde la columna 'MayorImpactoSustanciaId' y las columnas de sustancias especificadas contengan valores que estén en la lista de sustancias.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene los datos de las sustancias.
        sustancias (list): Lista de identificadores de sustancias a filtrar.
        excluir_mayor_impacto (list, optional): Lista de identificadores de sustancias a excluir de la columna 'MayorImpactoSustanciaId'. Por defecto es None.
    Retorna: Un DataFrame filtrado que contiene solo las filas relevantes.
    '''

    if excluir_mayor_impacto is None:
        excluir_mayor_impacto = []

    lista_columnas = [col for col in df.columns if col.startswith('SustanciaId')]

    # Sustancias válidas solo para MayorImpacto
    sustancias_mayor_impacto = [
        s for s in sustancias if s not in excluir_mayor_impacto
    ]

    mask = (
        df['MayorImpactoSustanciaId'].isin(sustancias_mayor_impacto)
        &
        df[lista_columnas].apply(
            lambda row: all((x in sustancias or pd.isnull(x)) for x in row),
            axis=1
        )
    )

    return df[mask]

def Tabaco_AlcoholUnicos(df ,sustancias):
    '''
    Descripcion: Esta funcion filtra un DataFrame para incluir solo las filas donde la columna 'MayorImpactoSustanciaId' y las columnas de sustancias especificadas contengan valores que estén en la lista de sustancias.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene los datos de las sustancias.
        sustancias (list): Lista de identificadores de sustancias a filtrar.
    Retorna: Un DataFrame filtrado que contiene solo las filas relevantes.
    '''
    df_legales = RecorridoSustancia(df, sustancias, excluir_mayor_impacto=['ProblemasDeSaludMental', 'Ninguna'])
    return df_legales

def Denominadores(df):
    '''
    Descripcion: Esta funcion crea tres DataFrames a partir de un DataFrame original:
    1. Universo: Contiene todos los registros del DataFrame original.
    2. DrogasLegaleseIlegales: Contiene registros donde 'MayorImpactoSustanciaId' no es 'ProblemasDeSaludMental'.
    3. DrogasIlegales: Contiene registros donde 'FolioId' no está en el DataFrame de tabaco y alcohol únicos, y 'MayorImpactoSustanciaId' no es 'ProblemasDeSaludMental'.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene los datos de las sustancias.
    Retorna: Tres DataFrames: Universo, DrogasLegaleseIlegales y DrogasIlegales.
    '''
    Universo = df
    # DrogasLegaleseIlegales = df[~df['MayorImpactoSustanciaId'].isin(['ProblemasDeSaludMental', 'Ninguna'])] #Esta seria la forma correcta si se usaran como droga de impacto para RIEGS
    DrogasLegaleseIlegales = df.copy()  #Actualmente se usa toda la muestra para RIEGS
    dfAlcoholTabaco = Tabaco_AlcoholUnicos(df, ['Tabaco', 'Alcohol'])
    DrogasIlegales = df[~df['FolioId'].isin(dfAlcoholTabaco['FolioId'])]
    # DrogasIlegales = DrogasIlegales[~DrogasIlegales['MayorImpactoSustanciaId'].isin(['ProblemasDeSaludMental', 'Ninguna'])]
    return Universo, DrogasLegaleseIlegales, DrogasIlegales

def DenominadoresSust(df):
    '''
    Descripcion: Esta funcion crea tres DataFrames a partir de un DataFrame original:
    1. Universo: Contiene todos los registros del DataFrame original.
    2. DrogasLegaleseIlegales: Contiene registros donde 'MayorImpactoSustanciaId' no es 'ProblemasDeSaludMental'.
    3. DrogasIlegales: Contiene registros donde 'FolioId' no está en el DataFrame de tabaco y alcohol únicos, y 'MayorImpactoSustanciaId' no es 'ProblemasDeSaludMental'.
    Parametros:
        df (DataFrame): DataFrame de pandas que contiene los datos de las sustancias.
    Retorna: Tres DataFrames: Universo, DrogasLegaleseIlegales y DrogasIlegales.
    '''
    Universo = df
    # DrogasLegaleseIlegales = df[~df['MayorImpactoSustanciaId'].isin(['Ninguna'])]
    DrogasLegaleseIlegales = df.copy()  #Actualmente se usa toda la muestra para RIEGS
    dfAlcoholTabaco = Tabaco_AlcoholUnicos(df, ['Tabaco','Nicotina en dispositivo electrónico','Bebidas Fermentadas','Bebidas Destiladas','Alcohol puro (alcohol del 96)'])
    DrogasIlegales = df[~df['FolioId'].isin(dfAlcoholTabaco['FolioId'])]
    # DrogasIlegales = DrogasIlegales[~DrogasIlegales['MayorImpactoSustanciaId'].isin(['ProblemasDeSaludMental', 'Ninguna'])]
    return Universo, DrogasLegaleseIlegales, DrogasIlegales

def cargar_datos(Universo, DrogasLegaleseIlegales, DrogasIlegales):
    '''
    Descripcion: Esta funcion sube tres DataFrames a un servidor FTP como archivos CSV.
    Parametros:
        Universo (DataFrame): DataFrame que contiene todos los registros.
        DrogasLegaleseIlegales (DataFrame): DataFrame que contiene registros de drogas legales e ilegales.
        DrogasIlegales (DataFrame): DataFrame que contiene registros de drogas ilegales.
    Retorna: None
    '''
    UploadCsvToSFTP(Universo, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialUniverso.csv')
    UploadCsvToSFTP(DrogasLegaleseIlegales, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasLegalesIlegales.csv')
    UploadCsvToSFTP(DrogasIlegales, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasIlegales.csv')

def cargar_datosSust(Universo, DrogasLegaleseIlegales, DrogasIlegales):
    '''
    Descripcion: Esta funcion sube tres DataFrames a un servidor FTP como archivos CSV, específicamente para datos de sustancias.
    Parametros:
        Universo (DataFrame): DataFrame que contiene todos los registros.
        DrogasLegaleseIlegales (DataFrame): DataFrame que contiene registros de drogas legales e ilegales.
        DrogasIlegales (DataFrame): DataFrame que contiene registros de drogas ilegales.
    Retorna: None
    '''
    UploadCsvToSFTP(Universo, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialUniversoSust.csv')
    UploadCsvToSFTP(DrogasLegaleseIlegales, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasLegalesIlegalesSust.csv')
    UploadCsvToSFTP(DrogasIlegales, r'riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasIlegalesSust.csv')

In [4]:
def main():
    try:
        EntrevistaInicial = ReadCsvFromSFTP(r'riecs/pruebas2/csv/Moddata/EntrevisaInicial.csv')
    except Exception as e:
        print("3)_Recod_data / Error al leer el archivo desde SFTP:", e)
        return
    try:
        EntrevistaInicial = Mod_SocioDem(EntrevistaInicial)
    except Exception as e:
        print("3)_Recod_data / Error al modificar datos socio-demográficos:", e)
        return    
    try:
        EntrevistaInicial, EntrevistaInicialsust = ModSustancias(EntrevistaInicial)
    except Exception as e:
        print("3)_Recod_data / Error al modificar datos de sustancias:", e)
        return
    try:
        Universo, DrogasLegaleseIlegales, DrogasIlegales= Denominadores(EntrevistaInicial)
        Universosust, DrogasLegaleseIlegalesust, DrogasIlegalesust= DenominadoresSust(EntrevistaInicialsust)
    except Exception as e:
        print("3)_Recod_data / Error al calcular denominadores:", e)
        return
    try:
        cargar_datos(Universo, DrogasLegaleseIlegales, DrogasIlegales)
        cargar_datosSust(Universosust, DrogasLegaleseIlegalesust, DrogasIlegalesust)
    except Exception as e:
        print("3)_Recod_data / Error al cargar datos al SFTP:", e)
        return
    return DrogasIlegales, DrogasIlegalesust

if __name__ == "__main__":
    df, dfsust = main()

Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_17652\2822184593.py:121: DtypeWarning: Columns (31,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,366,367,424,425) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


Conectado mediante SFTP a 192.168.62.150:22
Conectado mediante SFTP a 192.168.62.150:22
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialUniverso.csv
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasLegalesIlegales.csv
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasIlegales.csv
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialUniversoSust.csv
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasLegalesIlegalesSust.csv
Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EntrevistaInicialDrogasIlegalesSust.csv
